# Telco Customer Churn — Exploratory Data Analysis

**Goal:** Understand the dataset before training, validate preprocessing decisions, and identify the features most predictive of churn.

**Dataset:** [Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) — ~7,000 telecom customers, 21 columns.  
**Expected file:** `data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv`

---

### Sections
1. Setup & Load
2. Dataset Overview
3. Target Variable — Class Imbalance
4. Numeric Features
5. Tenure Cohorts
6. Categorical Features — Churn Rate by Category
7. Key Predictors Deep Dive
8. Correlation Analysis
9. Summary & Preprocessing Decisions

## 1. Setup & Load

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

CHURN_COLOR = "#DD8452"
RETAIN_COLOR = "#4C72B0"
PALETTE = [RETAIN_COLOR, CHURN_COLOR]

DATA_PATH = Path("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [ ]:
raw = pd.read_csv(DATA_PATH)

df = raw.copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].replace(" ", pd.NA), errors="coerce")
df.loc[df["TotalCharges"].isna() & (df["tenure"] == 0), "TotalCharges"] = 0.0
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
df.head(3)

## 2. Dataset Overview

In [ ]:
pd.DataFrame({
    "dtype": df.dtypes,
    "nulls": df.isnull().sum(),
    "null_%": (df.isnull().mean() * 100).round(2),
    "unique": df.nunique(),
    "sample": df.iloc[0],
})

In [ ]:
df.describe().T.style.background_gradient(cmap="Blues", subset=["mean", "std"])

> **Note on `TotalCharges`:** The raw CSV stores this column as a string. Eleven new customers (tenure = 0) have an empty string instead of `0`. This is fixed by the `clean_telco_data` function in `src/preprocess.py` before training.

## 3. Target Variable — Class Imbalance

In [ ]:
churn_rate = df["Churn"].mean()
churn_counts = df["Churn"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Count bar
bars = axes[0].bar(["Retained", "Churn"], churn_counts.values, color=PALETTE, edgecolor="white", linewidth=1.5)
for bar, count in zip(bars, churn_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 40,
                 f"{count:,}", ha="center", va="bottom", fontsize=12, fontweight="bold")
axes[0].set_title("Customer Count by Churn", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Customers")
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Pie
axes[1].pie(
    churn_counts.values,
    labels=["Retained", "Churn"],
    autopct="%1.1f%%",
    colors=PALETTE,
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    textprops={"fontsize": 12},
)
axes[1].set_title("Churn Rate", fontsize=13, fontweight="bold")

plt.suptitle(f"Overall churn rate: {churn_rate:.1%}  |  Imbalance ratio ≈ 3:1",
             fontsize=11, y=1.01, color="gray")
plt.tight_layout()
plt.show()

> **Imbalance (~26.5% churn):** The dataset is moderately imbalanced at roughly 3:1. This means:
> - **ROC-AUC** is the primary metric — it is insensitive to class distribution.
> - **F1-score** complements AUC by penalizing poor recall on the minority class.
> - Plain **accuracy** is misleading here (a model that predicts "no churn" always gets ~73%).
> - XGBoost handles this naturally via `scale_pos_weight`, though we keep the default for baseline.

## 4. Numeric Features

In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for col, (ax_hist, ax_box) in zip(numeric_cols, zip(axes[0], axes[1])):
    # Distribution by churn
    for churn_val, label, color in [(0, "Retained", RETAIN_COLOR), (1, "Churn", CHURN_COLOR)]:
        sns.kdeplot(
            df.loc[df["Churn"] == churn_val, col],
            ax=ax_hist, label=label, color=color, fill=True, alpha=0.35, linewidth=1.5,
        )
    ax_hist.set_title(col, fontsize=12, fontweight="bold")
    ax_hist.set_xlabel("")
    ax_hist.legend(fontsize=9)

    # Boxplot by churn
    sns.boxplot(
        data=df, x="Churn", y=col, ax=ax_box,
        palette=PALETTE, width=0.5, linewidth=1.2,
        flierprops={"marker": "o", "markersize": 3, "alpha": 0.4},
    )
    ax_box.set_xticklabels(["Retained", "Churn"])
    ax_box.set_xlabel("")
    ax_box.set_title(f"{col} by Churn", fontsize=11)

plt.suptitle("Numeric Feature Distributions", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

**Observations:**
- **`tenure`:** Churned customers skew heavily toward low tenure. Customers who stay longer are much less likely to churn — strong predictor.
- **`MonthlyCharges`:** Churned customers pay more per month. Suggests higher-tier plans correlate with churn.
- **`TotalCharges`:** Lower total charges for churned customers — consistent with short tenure (less time to accumulate charges).

## 5. Tenure Cohorts

In [ ]:
bins = [0, 12, 24, 36, 48, 60, 72]
labels = ["0–12m", "13–24m", "25–36m", "37–48m", "49–60m", "61–72m"]
df["tenure_cohort"] = pd.cut(df["tenure"], bins=bins, labels=labels, right=True)

cohort_stats = (
    df.groupby("tenure_cohort", observed=True)["Churn"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "churn_rate", "count": "n"})
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(cohort_stats["tenure_cohort"].astype(str), cohort_stats["churn_rate"],
              color=[CHURN_COLOR if r > 0.25 else RETAIN_COLOR for r in cohort_stats["churn_rate"]],
              edgecolor="white", linewidth=1.5)

for bar, row in zip(bars, cohort_stats.itertuples()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{row.churn_rate:.0%}\n(n={row.n:,})",
            ha="center", va="bottom", fontsize=9, color="#333")

ax.axhline(df["Churn"].mean(), linestyle="--", linewidth=1.5, color="gray", label=f"Overall avg ({df['Churn'].mean():.1%})")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title("Churn Rate by Tenure Cohort", fontsize=13, fontweight="bold")
ax.set_xlabel("Tenure Cohort")
ax.set_ylabel("Churn Rate")
ax.legend()
plt.tight_layout()
plt.show()

df.drop(columns=["tenure_cohort"], inplace=True)

> **Key insight:** The first 12 months are critical — new customers churn at nearly 2× the overall rate. After 2 years, churn drops dramatically. Retention programs should prioritize the 0–12m cohort.

## 6. Categorical Features — Churn Rate by Category

In [ ]:
categorical_cols = [
    "Contract", "InternetService", "PaymentMethod",
    "OnlineSecurity", "TechSupport", "OnlineBackup",
    "DeviceProtection", "StreamingTV", "StreamingMovies",
    "MultipleLines", "PaperlessBilling", "Partner",
    "Dependents", "PhoneService", "SeniorCitizen", "gender",
]

rows = []
for col in categorical_cols:
    for val, grp in df.groupby(col):
        rows.append({"feature": col, "value": str(val), "churn_rate": grp["Churn"].mean(), "n": len(grp)})

cat_stats = pd.DataFrame(rows)

fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes_flat = axes.flatten()

for i, col in enumerate(categorical_cols):
    ax = axes_flat[i]
    sub = cat_stats[cat_stats["feature"] == col].sort_values("churn_rate", ascending=True)
    colors = [CHURN_COLOR if r > df["Churn"].mean() else RETAIN_COLOR for r in sub["churn_rate"]]
    bars = ax.barh(sub["value"], sub["churn_rate"], color=colors, edgecolor="white")
    ax.axvline(df["Churn"].mean(), linestyle="--", linewidth=1, color="gray", alpha=0.7)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_title(col, fontsize=10, fontweight="bold")
    for bar, row in zip(bars, sub.itertuples()):
        ax.text(row.churn_rate + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{row.churn_rate:.0%}", va="center", fontsize=8)

# Hide unused subplot
axes_flat[-1].set_visible(False)

plt.suptitle("Churn Rate by Categorical Feature Value  (dashed = overall avg)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 7. Key Predictors — Deep Dive

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col in zip(axes, ["Contract", "InternetService"]):
    sub = df.groupby(col)["Churn"].agg(["mean", "count"]).reset_index()
    sub = sub.sort_values("mean", ascending=False)
    bars = ax.bar(sub[col], sub["mean"],
                  color=[CHURN_COLOR if r > df["Churn"].mean() else RETAIN_COLOR for r in sub["mean"]],
                  edgecolor="white", linewidth=1.5)
    for bar, row in zip(bars, sub.itertuples()):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.008,
                f"{row.mean:.0%}\n(n={row.count:,})",
                ha="center", va="bottom", fontsize=10)
    ax.axhline(df["Churn"].mean(), linestyle="--", linewidth=1.5, color="gray",
               label=f"Avg {df['Churn'].mean():.1%}")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_title(f"Churn Rate by {col}", fontsize=13, fontweight="bold")
    ax.set_ylabel("Churn Rate")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# PaymentMethod
sub_pay = df.groupby("PaymentMethod")["Churn"].agg(["mean", "count"]).reset_index().sort_values("mean", ascending=True)
colors = [CHURN_COLOR if r > df["Churn"].mean() else RETAIN_COLOR for r in sub_pay["mean"]]
bars = axes[0].barh(sub_pay["PaymentMethod"], sub_pay["mean"], color=colors, edgecolor="white")
axes[0].axvline(df["Churn"].mean(), linestyle="--", linewidth=1.5, color="gray")
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
for bar, row in zip(bars, sub_pay.itertuples()):
    axes[0].text(row.mean + 0.005, bar.get_y() + bar.get_height() / 2,
                 f"{row.mean:.0%}", va="center", fontsize=10)
axes[0].set_title("Churn Rate by Payment Method", fontsize=13, fontweight="bold")

# SeniorCitizen
senior_map = {0: "Non-Senior", 1: "Senior"}
sub_senior = df.assign(SeniorCitizen=df["SeniorCitizen"].map(senior_map)).groupby("SeniorCitizen")["Churn"].agg(["mean", "count"]).reset_index()
bars2 = axes[1].bar(sub_senior["SeniorCitizen"], sub_senior["mean"],
                    color=[RETAIN_COLOR, CHURN_COLOR], edgecolor="white", linewidth=1.5, width=0.4)
for bar, row in zip(bars2, sub_senior.itertuples()):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.008,
                 f"{row.mean:.0%}\n(n={row.count:,})", ha="center", va="bottom", fontsize=11)
axes[1].axhline(df["Churn"].mean(), linestyle="--", linewidth=1.5, color="gray")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
axes[1].set_title("Churn Rate: Senior vs Non-Senior", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

**Observations:**
- **Contract type** is the single strongest categorical predictor. Month-to-month customers churn at ~42% vs ~11% (one year) and ~3% (two year).
- **Internet service (Fiber optic)** customers churn at ~42% — possibly due to higher cost or service quality issues.
- **Electronic check** payment method has notably higher churn than automatic payment methods.
- **Senior citizens** churn at ~41% vs ~24% for non-seniors.

## 8. Correlation Analysis

In [ ]:
numeric_df = df[["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen", "Churn"]]

corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, square=True, ax=ax,
    annot_kws={"fontsize": 11},
)
ax.set_title("Numeric Feature Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import pointbiserialr, chi2_contingency

# Point-biserial correlation for numeric features
pb_rows = []
for col in ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]:
    r, p = pointbiserialr(df["Churn"], df[col])
    pb_rows.append({"feature": col, "type": "numeric", "correlation": r, "p_value": p})

# Cramér's V for categorical features
def cramers_v(col):
    ct = pd.crosstab(df[col], df["Churn"])
    chi2, p, _, _ = chi2_contingency(ct)
    n = ct.sum().sum()
    k = min(ct.shape) - 1
    return np.sqrt(chi2 / (n * k)), p

cat_features = [
    "Contract", "InternetService", "PaymentMethod",
    "OnlineSecurity", "TechSupport", "OnlineBackup",
    "DeviceProtection", "StreamingTV", "StreamingMovies",
    "MultipleLines", "PaperlessBilling", "Partner",
    "Dependents", "PhoneService", "gender",
]
for col in cat_features:
    v, p = cramers_v(col)
    pb_rows.append({"feature": col, "type": "categorical (Cramér V)", "correlation": v, "p_value": p})

assoc = (
    pd.DataFrame(pb_rows)
    .sort_values("correlation", key=abs, ascending=False)
    .reset_index(drop=True)
)
assoc["significant"] = assoc["p_value"] < 0.05
assoc.style.background_gradient(cmap="Oranges", subset=["correlation"]).format({"correlation": "{:.3f}", "p_value": "{:.2e}"})

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

colors = [CHURN_COLOR if t == "numeric" else RETAIN_COLOR for t in assoc["type"]]
bars = ax.barh(assoc["feature"], assoc["correlation"].abs(), color=colors, edgecolor="white")

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=RETAIN_COLOR, label="Categorical (Cramér\'s V)"),
    Patch(facecolor=CHURN_COLOR, label="Numeric (Point-biserial r)"),
]
ax.legend(handles=legend_elements, fontsize=10)
ax.set_title("Feature Association with Churn (sorted by strength)", fontsize=13, fontweight="bold")
ax.set_xlabel("Association strength")
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 9. Summary & Preprocessing Decisions

### Key Findings

| Finding | Implication |
|---|---|
| Churn rate ≈ 26.5% (3:1 imbalance) | Use ROC-AUC + F1 as primary metrics, not accuracy |
| `tenure` is the strongest numeric predictor | Keep as-is; model captures the non-linear decay |
| `Contract` has the highest categorical association | One-hot encoding will let the model distinguish month-to-month clearly |
| `TotalCharges` = f(tenure × MonthlyCharges) | Multicollinearity exists, but XGBoost handles it without issue |
| 11 rows have `TotalCharges` = NaN (new customers) | `clean_telco_data` imputes 0.0 using the tenure == 0 heuristic |
| `gender` has no significant association with churn | Expected; included for completeness but will have low feature importance |

### Preprocessing Pipeline Decisions (src/preprocess.py)

- **Numeric features** → median imputation (robust to outliers in `MonthlyCharges`).
- **Categorical features** → most-frequent imputation + `OneHotEncoder(handle_unknown="ignore")`. The `handle_unknown` guard ensures the API does not crash if a future category value is not seen in training.
- **`customerID` dropped** — identifier with no predictive signal.
- **No scaling applied** — XGBoost is tree-based and invariant to feature scale.
- **`make_column_selector`** used instead of hardcoded column lists — this makes the preprocessor resilient to column reordering.